# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pathlib
import os
from dotenv import load_dotenv
import isodate
import datetime

def get_isodatetime():
    """get the current time as an iso datetime"""
    current = datetime.datetime.now(tz=datetime.timezone.utc)
    iso_value = isodate.datetime_isoformat(current)
    return iso_value

def get_isodate():
    """get the current time as an iso datetime"""
    current = datetime.datetime.now(tz=datetime.timezone.utc)
    iso_value = isodate.date_isoformat(current)
    return iso_value
#local functions
import video_extraction
import audio_transcription
import transcript_merger
import summarize_transcript

# Load environment variables
load_dotenv()
logger = video_extraction.logger

## Transcribe

In [ ]:
# Args
#-- extraction
dir_subfolder="client" #change for subfolder export. "" is none
input_mkv = "./+recordings/2024-07-17_15-29-04-meetingnotes.mkv"

# -- transcribing
mic_audio_filename = f"microphone_audio.wav"
desktop_audio_filename = f"desktop_audio.wav"
# -- processing/merging
mic_transcript_filename = f"microphone_transcription.txt"
desktop_transcript_filename = f"desktop_transcription.txt"
transcript_filename = f"merged_transcript.json"
# -- summarizing
summary_filename=f"summary.txt"
summarize_model_name = 'claude-3-5-sonnet'

# Script Code
# - Setup output directories
output_dir = f"./{get_isodate()}-output"
output_path = pathlib.Path(output_dir) if dir_subfolder in ["",None,False] else pathlib.Path(output_dir) / dir_subfolder
if not output_path.exists():
    output_path.mkdir(parents=True)
    
print(f"Output dir is: {output_path}")
# -- Extract audio files
try:
   result = video_extraction.process_mkv_file(input_mkv, str(output_path))
except Exception as e:
    logger.error(f"An error occurred during the extraction process: {str(e)}")
    
# -- Transcribe audio
mic_audio_path = output_path / mic_audio_filename
desktop_audio_path = output_path / desktop_audio_filename
try:
    filenames = audio_transcription.process_audio_files(str(mic_audio_path), str(desktop_audio_path), output_dir=str(output_path))
    logger.info(filenames)
except Exception as e:
    logger.error(f"An error occurred during the transcription process: {str(e)}")

# -- Merge and process transcription
mic_transcript_path = filenames['microphone']
desktop_transcript_path = filenames['desktop']
transcript_file_path = output_path / transcript_filename
try:
    transcript_json = transcript_merger.process_transcripts(
        mic_transcript_file=str(mic_transcript_path), 
        desktop_transcript_file=str(desktop_transcript_path), 
        output_file=str(transcript_file_path),
        parse_overlaps=False
    )
except Exception as e:
    logger.error(f"An error occurred during the merging process: {str(e)}")

# -- Augment with additional information
# TODO: add personal notes taken during meeting
# TODO: add other md, txt, or other text files


# -- Summarize
# Main workflow
summary_filepath = output_path / summary_filename

transcript_data = summarize_transcript.load_transcript(str(transcript_file_path))
transcript_text = summarize_transcript.prepare_transcript_for_summary(transcript_data)
# Estimate tokens and price
total_tokens, estimated_price = summarize_transcript.estimate_tokens_and_price(
    transcript_text, 
    model=summarize_model_name
)

# summarize=False #True to call the llm
# # if summarize:
# #     try:
#         summary = summarize_transcript.generate_summary(transcript_text, model_name=summarize_model_name)
#         summarize_transcript.save_summary(
#             summary, 
#             output_file=str(summary_filepath)
#         )
#         logger.info(f"Summary generated and saved to {summary_filepath}")
#     except Exception as e:
#         logger.error(f"An error occurred during summarization: {str(e)}")
#     logger.info(summary)
# else:
#     logger.info("Skipping summarize step")

## Summarize

In [ ]:
def export_to_markdown(content, filename):
    with open(filename, 'w', encoding='utf-8') as file:
        file.write(content)
        
summarize = True

# -- Summarize
# Main workflow
summary_filepath = output_path / f"{get_isodatetime()}-{summary_filename}"
transcript_data = summarize_transcript.load_transcript(str(transcript_file_path))
transcript_text = summarize_transcript.prepare_transcript_for_summary(transcript_data)
# Estimate tokens and price
total_tokens, estimated_price = summarize_transcript.estimate_tokens_and_price(
    transcript_text, 
    model=summarize_model_name
)
logger.info(f"Summary filename: {summary_filepath}")
if summarize:
    try:
        summary = summarize_transcript.generate_summary(transcript_text, model_name=summarize_model_name)
        summarize_transcript.save_summary(
            summary, 
            output_file=str(summary_filepath)
        )
        logger.info(f"Summary generated and saved to {summary_filepath}")
        export_to_markdown(summary, filename=str(summary_filepath.with_suffix('.md')))
    except Exception as e:
        logger.error(f"An error occurred during summarization: {str(e)}")
else:
    logger.info("Skipping summarize step")

In [ ]:
import json
import os
import sys
import logging
from dotenv import load_dotenv
from anthropic import Anthropic, HUMAN_PROMPT, AI_PROMPT
import tiktoken

import summarize_transcript

model_name = 'claude-3-5-sonnet'
MAX_TOKENS_OUTPUT=None

summary_filepath = output_path / summary_filename
transcript_data = summarize_transcript.load_transcript(str(transcript_file_path))
transcript_text = summarize_transcript.prepare_transcript_for_summary(transcript_data)
# Estimate tokens and price
total_tokens, estimated_price = summarize_transcript.estimate_tokens_and_price(
    transcript_text, 
    model=model_name,
    max_output_tokens=MAX_TOKENS_OUTPUT
)


    
if model_name not in summarize_transcript.anthropic_models:
    raise KeyError(f"model must be one of [{list(anthropic_models.keys())}]")
model = summarize_transcript.anthropic_models.get(model_name)['name']
logger.info(f"Will summarize using: {model}")
client = Anthropic(api_key=summarize_transcript.ANTHROPIC_API_KEY)
prompt = summarize_transcript.create_summary_prompt(transcript_text)

In [ ]:
 try:
        response = client.messages.create(
            model=model,
            max_tokens=MAX_TOKENS_OUTPUT,
            temperature=0.7,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        return response.content[0].text
    except Exception as e:
        logger.error(f"Error generating summary: {str(e)}")
        raise

In [ ]:
#extract audio files
input_mkv = "./data/test.mkv"
output_dir = "./output"
result = video_extraction.process_mkv_file(input_mkv, output_dir)

In [ ]:
import transcript_merger
# Example usage
mic_transcript_file = './output/microphone_transcription.txt'
desktop_transcript_file = './output/desktop_transcription.txt'
output_file = './output/merged_transcript.json'

transcript_json = transcript_merger.process_transcripts(
    mic_transcript_file, 
    desktop_transcript_file, 
    output_file,
    parse_overlaps=False
)